# 🌲 Fase 1 — Entrenamiento y Evaluación: Random Forest
Objetivo: Entrenar RF con vectores HOG y evaluar con F1, AUC-ROC y Matriz de Confusión.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
    f1_score, roc_auc_score, accuracy_score, roc_curve)
SEED=42; np.random.seed(SEED)
print('✅ Librerías OK')

In [ ]:
ROOT=Path('..').resolve()
PROC=ROOT/'data'/'processed'
MODELS=ROOT/'results'/'models'; MODELS.mkdir(parents=True,exist_ok=True)
FIGS=ROOT/'results'/'figures';  FIGS.mkdir(parents=True,exist_ok=True)
METS=ROOT/'results'/'metrics';  METS.mkdir(parents=True,exist_ok=True)

X_train=np.load(PROC/'X_train.npy'); y_train=np.load(PROC/'y_train.npy')
X_val  =np.load(PROC/'X_val.npy');   y_val  =np.load(PROC/'y_val.npy')
X_test =np.load(PROC/'X_test.npy');  y_test =np.load(PROC/'y_test.npy')
print(f'Train:{X_train.shape} Val:{X_val.shape} Test:{X_test.shape}')

In [ ]:
rf = RandomForestClassifier(n_estimators=200,max_depth=None,min_samples_split=5,
    min_samples_leaf=2,class_weight='balanced',random_state=SEED,n_jobs=-1)
print('🌲 Entrenando...'); rf.fit(X_train,y_train); print('✅ Entrenamiento completado')

In [ ]:
y_pred=rf.predict(X_val); y_proba=rf.predict_proba(X_val)[:,1]
print(f'Accuracy : {accuracy_score(y_val,y_pred):.4f}')
print(f'F1-Score : {f1_score(y_val,y_pred,average="weighted"):.4f}')
print(f'AUC-ROC  : {roc_auc_score(y_val,y_proba):.4f}')
print(); print(classification_report(y_val,y_pred,target_names=['Asfalto Normal','Bache/Rejilla']))

In [ ]:
cm=confusion_matrix(y_val,y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',
    xticklabels=['Asfalto Normal','Bache/Rejilla'],yticklabels=['Asfalto Normal','Bache/Rejilla'])
plt.title('Matriz de Confusión — RF Fase 1',fontweight='bold')
plt.ylabel('Real'); plt.xlabel('Predicho')
plt.tight_layout(); plt.savefig(FIGS/'confusion_matrix_rf.png',dpi=150); plt.show()

In [ ]:
fpr,tpr,_=roc_curve(y_val,y_proba); auc=roc_auc_score(y_val,y_proba)
plt.figure(figsize=(8,6))
plt.plot(fpr,tpr,'darkorange',lw=2,label=f'AUC={auc:.4f}')
plt.plot([0,1],[0,1],'navy',lw=1,linestyle='--')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('Curva ROC — RF Fase 1',fontweight='bold')
plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIGS/'roc_curve_rf.png',dpi=150); plt.show()

In [ ]:
y_tp=rf.predict(X_test); y_tproba=rf.predict_proba(X_test)[:,1]
res={'modelo':'Random Forest','fase':1,
     'accuracy':round(accuracy_score(y_test,y_tp),4),
     'f1_score':round(f1_score(y_test,y_tp,average='weighted'),4),
     'auc_roc':round(roc_auc_score(y_test,y_tproba),4)}
pd.DataFrame([res]).to_csv(METS/'metricas_fase1.csv',index=False)
joblib.dump(rf, MODELS/'random_forest_fase1.pkl')
print('✅ Métricas y modelo guardados')
[print(f'{k:12s}: {v}') for k,v in res.items()]